In [ ]:
import geopandas as gpd
import pandas as pd
import folium

In [ ]:
# read files
sf = gpd.read_file("../data/landing/sa2/SA2_2021_AUST_GDA2020.shp")
income_data = pd.read_csv("../data/curated/suburb_info.csv")

sf.head()

In [ ]:
# only keep victorian SA2 codes
sf = sf.loc[sf["STE_NAME21"] == "Victoria"]
sf['SA2_CODE21'] = sf['SA2_CODE21'].astype(int)

In [ ]:
gdf = gpd.GeoDataFrame(
    income_data.merge(sf, right_on='SA2_CODE21', left_on='SA2_codes', how='left')
)

gdf.head()

In [ ]:
# create a JSON 
geoJSON = gdf[['SA2_codes', 'geometry']].drop_duplicates('SA2_codes').to_json()

# print the first 300 chars of the json
print(geoJSON[:300])

In [ ]:
df = gdf[["SA2_codes", 'Median equivalised total household income (weekly) ($)']]

df.head()

# Income map of Victoria with top vce schools pointers

In [ ]:
m = folium.Map(location=[-37.020100, 144.964600], tiles="Stamen Terrain", zoom_start=7)

# refer to the folium documentations on more information on how to plot aggregated data.
c = folium.Choropleth(
    geo_data=geoJSON, # geoJSON 
    name='choropleth', # name of plot
    data=df.reset_index(), # data source
    columns=['SA2_codes','Median equivalised total household income (weekly) ($)'], # the columns required
    key_on= 'properties.SA2_codes', # this is from the geoJSON's properties
    fill_color='YlOrRd', # color scheme
    nan_fill_color='black',
    legend_name='Median equivalised total household income (weekly) ($)'
)

c.add_to(m)

In [ ]:
school_data = pd.read_csv("../data/curated/school_data/school_data.csv")
school_data["coordinate"] = school_data["coordinate"].apply(eval)
# top vce schools 
top_secondary_schools = school_data[school_data["median vce score"].notnull()]

In [ ]:
# insert pointers to top schools on map
for school_name, coord in top_secondary_schools[["school_name", "coordinate"]].values:
    m.add_child(
        folium.Marker(location=coord, popup=school_name)
    )
    


In [ ]:
m.save('../plots/income_school_geo.html')

In [ ]:
m